In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import sys

project_path = r'C:\Users\VISHNU\Downloads\nifty100_project'
sys.path.append(project_path)
os.chdir(project_path)

from src.etl.loader import load_all_data

data = load_all_data()
market_cap = data['market_cap']
sectors = data['sectors']

# Load financial ratios
conn = sqlite3.connect('data/nifty100.db')
ratios = pd.read_sql_query(
    "SELECT * FROM financial_ratios_computed", conn
)
conn.close()

print(f"Market cap: {market_cap.shape}")
print(f"Ratios: {ratios.shape}")
print("Setup complete!")

Loading all datasets...

Dataset Summary:
  Dataset                Rows   Cols
  -----------------------------------
  profitandloss          1164     15
  balancesheet           1165     13
  cashflow               1152      7
  companies                92     12
  analysis                 20      6
  documents              1585      4
  prosandcons              16      4
  sectors                  92      6
  market_cap              552      9
  financial_ratios       1184     16
  peer_groups              56      4

All datasets loaded and cleaned successfully!
Market cap: (552, 9)
Ratios: (1159, 43)
Setup complete!


In [2]:
ANALYSIS_YEAR = '2024-03'
MARKET_CAP_YEAR = 2024

# Latest ratios
latest_ratios = ratios[ratios['year'] == ANALYSIS_YEAR].copy()

# Latest market cap
latest_mc = market_cap[market_cap['year'] == MARKET_CAP_YEAR][[
    'company_id', 'market_cap_crore', 'pe_ratio', 'pb_ratio', 'ev_ebitda'
]].copy()

# Merge
val_df = pd.merge(latest_ratios, latest_mc, on='company_id', how='left')

# Merge sectors
val_df = pd.merge(
    val_df,
    sectors[['company_id', 'broad_sector']],
    on='company_id', how='left'
)

print(f"Valuation DataFrame: {val_df.shape}")
print(f"Companies with all data: {val_df.dropna(subset=['free_cash_flow_cr', 'pe_ratio']).shape[0]}")

Valuation DataFrame: (98, 48)
Companies with all data: 90


In [3]:
def compute_fcf_yield(df):
    """
    FCF Yield = (Free Cash Flow / Market Cap) × 100
    
    Interpretation:
    > 10% = Excellent (cheap, high cash generation)
    5-10% = Good
    2-5%  = Fair
    < 2%  = Expensive (low cash return)
    """
    df = df.copy()
    
    df['fcf_yield_pct'] = np.where(
        (df['free_cash_flow_cr'].notna()) & (df['market_cap_crore'] > 0),
        (df['free_cash_flow_cr'] / df['market_cap_crore'] * 100).round(2),
        np.nan
    )
    
    # FCF Yield band
    def get_fcf_band(yield_pct):
        if pd.isna(yield_pct):
            return 'N/A'
        elif yield_pct > 10:
            return 'Excellent'
        elif yield_pct > 5:
            return 'Good'
        elif yield_pct > 2:
            return 'Fair'
        else:
            return 'Low'
    
    df['fcf_yield_band'] = df['fcf_yield_pct'].apply(get_fcf_band)
    
    return df

val_df = compute_fcf_yield(val_df)

print("FCF Yield Distribution:")
print(val_df['fcf_yield_band'].value_counts().to_string())
print()
print("Top 10 companies by FCF Yield:")
print(val_df.nlargest(10, 'fcf_yield_pct')[
    ['company_id', 'free_cash_flow_cr', 'market_cap_crore', 'fcf_yield_pct', 'fcf_yield_band']
].to_string(index=False))

FCF Yield Distribution:
fcf_yield_band
Low          77
N/A           8
Fair          6
Good          4
Excellent     3

Top 10 companies by FCF Yield:
company_id  free_cash_flow_cr  market_cap_crore  fcf_yield_pct fcf_yield_band
       TCS            50429.0         461188.15          10.93      Excellent
      ONGC            42060.0         390111.05          10.78      Excellent
  RELIANCE            45207.0         432472.71          10.45      Excellent
BHARTIARTL            27809.0         378605.83           7.35           Good
 POWERGRID            24176.0         372584.24           6.49           Good
       ITC            18742.0         332245.58           5.64           Good
      BPCL            25415.0         468662.13           5.42           Good
  HDFCBANK            35669.0         808359.03           4.41           Fair
   HCLTECH            15840.0         412593.29           3.84           Fair
 COALINDIA            13617.0         454024.84           3.00       

In [4]:
def compute_sector_pe_metrics(df):
    """
    Compares each company's P/E to its sector median.
    Flags overvalued (Caution) and undervalued (Discount) companies.
    """
    df = df.copy()
    
    # Compute sector median P/E
    sector_pe_median = df.groupby('broad_sector')['pe_ratio'].median()
    df['sector_median_pe'] = df['broad_sector'].map(sector_pe_median)
    
    # P/E vs sector median %
    df['pe_vs_sector_pct'] = np.where(
        df['sector_median_pe'].notna() & (df['sector_median_pe'] > 0),
        ((df['pe_ratio'] - df['sector_median_pe']) / df['sector_median_pe'] * 100).round(1),
        np.nan
    )
    
    # Valuation flag
    def get_valuation_flag(pe, sector_pe):
        if pd.isna(pe) or pd.isna(sector_pe):
            return 'N/A'
        
        ratio = pe / sector_pe
        if ratio > 1.5:
            return 'Caution'  # Expensive
        elif ratio < 0.7:
            return 'Discount'  # Cheap
        else:
            return 'Fair'
    
    df['valuation_flag'] = df.apply(
        lambda row: get_valuation_flag(row['pe_ratio'], row['sector_median_pe']),
        axis=1
    )
    
    return df

val_df = compute_sector_pe_metrics(val_df)

print("Valuation Flags Distribution:")
print(val_df['valuation_flag'].value_counts().to_string())
print()
print("Companies flagged as DISCOUNT (cheap):")
discount = val_df[val_df['valuation_flag'] == 'Discount'][
    ['company_id', 'broad_sector', 'pe_ratio', 'sector_median_pe', 'pe_vs_sector_pct', 'valuation_flag']
]
print(f"Total: {len(discount)}")
if len(discount) > 0:
    print(discount.to_string(index=False))
print()
print("Companies flagged as CAUTION (expensive):")
caution = val_df[val_df['valuation_flag'] == 'Caution'][
    ['company_id', 'broad_sector', 'pe_ratio', 'sector_median_pe', 'pe_vs_sector_pct', 'valuation_flag']
]
print(f"Total: {len(caution)}")
if len(caution) > 0:
    print(caution.head(10).to_string(index=False))

Valuation Flags Distribution:
valuation_flag
Fair        48
Discount    30
Caution     13
N/A          7

Companies flagged as DISCOUNT (cheap):
Total: 30
company_id           broad_sector  pe_ratio  sector_median_pe  pe_vs_sector_pct valuation_flag
BAJAJFINSV             Financials     15.07            34.240             -56.0       Discount
BANKBARODA             Financials     16.89            34.240             -50.7       Discount
       BEL            Industrials     17.89            48.900             -63.4       Discount
      BHEL            Industrials     29.80            48.900             -39.1       Discount
  CHOLAFIN             Financials     16.79            34.240             -51.0       Discount
 COALINDIA              Materials     34.27            55.620             -38.4       Discount
   DRREDDY             Healthcare     11.73            47.060             -75.1       Discount
 EICHERMOT Consumer Discretionary     15.62            38.005             -58.9      

In [5]:
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment

# Prepare output columns
output_cols = [
    'company_id', 'broad_sector',
    'pe_ratio', 'pb_ratio', 'ev_ebitda',
    'free_cash_flow_cr', 'market_cap_crore',
    'fcf_yield_pct', 'fcf_yield_band',
    'sector_median_pe', 'pe_vs_sector_pct', 'valuation_flag'
]

val_export = val_df[output_cols].copy()
val_export = val_export.sort_values('fcf_yield_pct', ascending=False, na_position='last')

# Save to Excel
val_export.to_excel('output/valuation_summary.xlsx', index=False)

print("valuation_summary.xlsx saved!")
print(f"Rows: {len(val_export)}")
print()
print("Sample (Top 5 by FCF Yield):")
print(val_export.head(5).to_string(index=False))

valuation_summary.xlsx saved!
Rows: 98

Sample (Top 5 by FCF Yield):
company_id           broad_sector  pe_ratio  pb_ratio  ev_ebitda  free_cash_flow_cr  market_cap_crore  fcf_yield_pct fcf_yield_band  sector_median_pe  pe_vs_sector_pct valuation_flag
       TCS Information Technology     78.69      6.08      29.94            50429.0         461188.15          10.93      Excellent            63.630              23.7           Fair
      ONGC                 Energy     18.15     10.36      11.72            42060.0         390111.05          10.78      Excellent            44.915             -59.6       Discount
  RELIANCE                 Energy     58.34      3.25      24.96            45207.0         432472.71          10.45      Excellent            44.915              29.9           Fair
BHARTIARTL Communication Services     63.78      6.28      39.71            27809.0         378605.83           7.35           Good            48.330              32.0           Fair
 POWERGRID      

In [6]:
print("Sprint 4 Day 2 — Valuation Summary:")
print()
print(f"Companies analyzed: {val_df.dropna(subset=['pe_ratio']).shape[0]}")
print()
print("FCF Yield Statistics:")
print(val_df['fcf_yield_pct'].describe().round(2).to_string())
print()
print("Valuation Flags:")
print(f"  Fair:     {(val_df['valuation_flag'] == 'Fair').sum()} companies")
print(f"  Discount: {(val_df['valuation_flag'] == 'Discount').sum()} companies")
print(f"  Caution:  {(val_df['valuation_flag'] == 'Caution').sum()} companies")
print()
print("Best FCF Yield (Top 5):")
print(val_df.nlargest(5, 'fcf_yield_pct')[
    ['company_id', 'fcf_yield_pct', 'valuation_flag']
].to_string(index=False))

Sprint 4 Day 2 — Valuation Summary:

Companies analyzed: 91

FCF Yield Statistics:
count    90.00
mean      0.36
std       3.69
min     -20.40
25%      -0.09
50%       0.14
75%       0.98
max      10.93

Valuation Flags:
  Fair:     48 companies
  Discount: 30 companies
  Caution:  13 companies

Best FCF Yield (Top 5):
company_id  fcf_yield_pct valuation_flag
       TCS          10.93           Fair
      ONGC          10.78       Discount
  RELIANCE          10.45           Fair
BHARTIARTL           7.35           Fair
 POWERGRID           6.49           Fair
